# Hausa Sentiment - Step 3: RNN experiment

This optional neural baseline compares an embedding model with BiLSTM models. Selection uses `dev_noleak`; the official test set is evaluated only for the selected configuration. Run in Google Colab (GPU recommended).


In [1]:
!pip -q install pandas numpy scikit-learn torch
from pathlib import Path
import json, random, time
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
from torch import nn
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
required = ["afrisenti_hau_train.csv", "afrisenti_hau_dev_noleak.csv", "afrisenti_hau_test.csv"]
candidates = [Path("/content/hausa-sentiment"), Path("/content"), Path.cwd(), Path.cwd().parent]
ROOT = PROC = None
for root in candidates:
    for proc in (root / "data" / "processed", root):
        if all((proc / name).exists() for name in required):
            ROOT, PROC = root, proc
            break
    if PROC is not None:
        break
if PROC is None:
    raise FileNotFoundError("Processed AfriSenti CSVs were not found in the Colab or repository layouts.")
MODELS, RESULTS = ROOT / "models", ROOT / "results"
MODELS.mkdir(parents=True, exist_ok=True); RESULTS.mkdir(parents=True, exist_ok=True)
print("device:", DEVICE, "\nPROC:", PROC)


device: cpu 
PROC: C:\Users\hp\Downloads\hausa-sentiment-v2\hausa-sentiment-v2\data\processed


In [2]:
train = pd.read_csv(PROC / "afrisenti_hau_train.csv")
dev = pd.read_csv(PROC / "afrisenti_hau_dev_noleak.csv")
test = pd.read_csv(PROC / "afrisenti_hau_test.csv")
MAX_LEN, MIN_FREQ, MAX_VOCAB = 64, 2, 30000
counts = {}
for text in train["text_bow"].fillna(""):
    for token in str(text).split(): counts[token] = counts.get(token, 0) + 1
tokens = sorted((t for t, c in counts.items() if c >= MIN_FREQ), key=lambda t: (-counts[t], t))[:MAX_VOCAB - 2]
stoi = {"<pad>": 0, "<unk>": 1}; stoi.update({t: i + 2 for i, t in enumerate(tokens)})

def encode(text):
    ids = [stoi.get(t, 1) for t in str(text).split()[:MAX_LEN]]
    mask = [1] * len(ids)
    ids += [0] * (MAX_LEN - len(ids)); mask += [0] * (MAX_LEN - len(mask))
    return ids, mask

class HausaDataset(Dataset):
    def __init__(self, frame):
        values = [encode(x) for x in frame["text_bow"].fillna("")]
        self.ids = torch.tensor([x[0] for x in values], dtype=torch.long)
        self.mask = torch.tensor([x[1] for x in values], dtype=torch.bool)
        self.labels = torch.tensor(frame["label_id"].astype(int).to_numpy(), dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.ids[i], self.mask[i], self.labels[i]

datasets = {"train": HausaDataset(train), "dev": HausaDataset(dev), "test": HausaDataset(test)}
print(f"rows: {len(train)} train, {len(dev)} dev_noleak, {len(test)} test; vocab={len(stoi)}")


rows: 14172 train, 2462 dev_noleak, 5303 test; vocab=10654


In [3]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, pooling):
        super().__init__(); self.pooling = pooling
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.rnn = None if pooling == "embedding_mean" else nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        output_dim = embedding_dim if self.rnn is None else hidden_dim * 2
        self.attention = nn.Linear(output_dim, 1) if pooling == "attention" else None
        self.dropout = nn.Dropout(0.3); self.classifier = nn.Linear(output_dim, 3)
    def forward(self, ids, mask):
        features = self.embedding(ids)
        if self.rnn is not None: features, _ = self.rnn(features)
        valid = mask.unsqueeze(-1)
        if self.pooling in ("embedding_mean", "mean"):
            pooled = (features * valid).sum(1) / valid.sum(1).clamp_min(1)
        else:
            scores = self.attention(features).masked_fill(~valid, -1e4)
            pooled = (features * torch.softmax(scores, dim=1)).sum(1)
        return self.classifier(self.dropout(pooled))

def evaluate(model, loader):
    model.eval(); truth, predictions = [], []
    with torch.no_grad():
        for ids, mask, labels in loader:
            predictions.extend(model(ids.to(DEVICE), mask.to(DEVICE)).argmax(1).cpu().numpy())
            truth.extend(labels.numpy())
    return {"accuracy": float(accuracy_score(truth, predictions)), "macro_f1": float(f1_score(truth, predictions, average="macro")), "f1_positive": float(f1_score(truth, predictions, labels=[0], average="macro")), "f1_neutral": float(f1_score(truth, predictions, labels=[1], average="macro")), "f1_negative": float(f1_score(truth, predictions, labels=[2], average="macro"))}

def train_one(config):
    batch = 128; train_loader = DataLoader(datasets["train"], batch_size=batch, shuffle=True); dev_loader = DataLoader(datasets["dev"], batch_size=batch)
    model = RNNClassifier(len(stoi), config["embedding_dim"], config["hidden_dim"], config["pooling"]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=1e-4); criterion = nn.CrossEntropyLoss()
    best_state, best_score, stale = None, -1.0, 0; started = time.time()
    for epoch in range(1, config["epochs"] + 1):
        model.train()
        for ids, mask, labels in train_loader:
            optimizer.zero_grad(); loss = criterion(model(ids.to(DEVICE), mask.to(DEVICE)), labels.to(DEVICE)); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
        metrics = evaluate(model, dev_loader); print(config["id"], "epoch", epoch, "dev_macro_f1", round(metrics["macro_f1"], 4))
        if metrics["macro_f1"] > best_score:
            best_score = metrics["macro_f1"]; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}; stale = 0
        else:
            stale += 1
            if stale >= config["patience"]: break
    model.load_state_dict(best_state); return model, time.time() - started


In [4]:
configs = [
    {"id": "R1", "name": "trainable embedding + mean pooling", "pooling": "embedding_mean", "embedding_dim": 128, "hidden_dim": 128, "lr": 2e-3, "epochs": 8, "patience": 2},
    {"id": "R2", "name": "BiLSTM + mean pooling", "pooling": "mean", "embedding_dim": 128, "hidden_dim": 128, "lr": 1e-3, "epochs": 8, "patience": 2},
    {"id": "R3", "name": "BiLSTM + attention pooling", "pooling": "attention", "embedding_dim": 128, "hidden_dim": 128, "lr": 1e-3, "epochs": 8, "patience": 2},
]
dev_loader = DataLoader(datasets["dev"], batch_size=128); registry = []; fitted = {}
for config in configs:
    model, seconds = train_one(config); metrics = evaluate(model, dev_loader)
    registry.append({**config, "dev_noleak": metrics, "train_seconds": round(seconds, 2)}); fitted[config["id"]] = model
best = max(registry, key=lambda row: row["dev_noleak"]["macro_f1"]); best_model = fitted[best["id"]]
best["test"] = evaluate(best_model, DataLoader(datasets["test"], batch_size=128)); best["n_params"] = sum(p.numel() for p in best_model.parameters())
with open(RESULTS / "rnn_registry.json", "w") as f: json.dump(registry, f, indent=2)
with open(RESULTS / "rnn_best_config.json", "w") as f: json.dump(best, f, indent=2)
pd.DataFrame([{ "id": r["id"], "name": r["name"], "dev_macro_f1": r["dev_noleak"]["macro_f1"], "train_seconds": r["train_seconds"] } for r in registry]).sort_values("dev_macro_f1", ascending=False).to_csv(RESULTS / "rnn_experiments.csv", index=False)
torch.save({"state_dict": best_model.state_dict(), "vocab": stoi, "config": best}, MODELS / "rnn_best.pt")
print("selected:", best["id"], best["name"]); print(json.dumps({"dev_noleak": best["dev_noleak"], "test": best["test"], "n_params": best["n_params"]}, indent=2))


R1 epoch 1 dev_macro_f1 0.5514


R1 epoch 2 dev_macro_f1 0.5852


R1 epoch 3 dev_macro_f1 0.6364


R1 epoch 4 dev_macro_f1 0.6443


R1 epoch 5 dev_macro_f1 0.6576


...

R3 epoch 3 dev_macro_f1 0.652


R3 epoch 4 dev_macro_f1 0.662


R3 epoch 5 dev_macro_f1 0.6558


R3 epoch 6 dev_macro_f1 0.662


selected: R1 trainable embedding + mean pooling
{
  "dev_noleak": {
    "accuracy": 0.6811535337124289,
    "macro_f1": 0.6824956379201286,
    "f1_positive": 0.7989949748743719,
    "f1_neutral": 0.6321070234113713,
    "f1_negative": 0.6163849154746424
  },
  "test": {
    "accuracy": 0.6831981897039412,
    "macro_f1": 0.6813171778267632,
    "f1_positive": 0.7899915659263425,
    "f1_neutral": 0.5952235150030618,
    "f1_negative": 0.6587364525508855
  },
  "n_params": 1364099
}
